# Day 11: Text Embeddings and Cosine Similarity

Welcome to Day 11 of the AI Engineering Mastery program. Today, we transition from pure text generation into how machines actually *understand* text: **Embeddings**. 

In this module, we will explore:
1. **Core Theory**: What are embeddings and why do we need them? How do we measure similarity?
2. **Code Implementation**: Generating embeddings using `sentence-transformers` and computing cosine similarity.
3. **Common Pitfalls**: What breaks in production when working with embeddings.
4. **Practical Lab**: An actionable task to build your own semantic search function.


## 1. Core Theory (Just-in-Time)

### The "Why": Machines Don't Read Text
Language Models and search systems cannot fundamentally process strings of text like "apple" or "I love programming." They require mathematical representations of these strings. An **embedding** is a translation of text into a high-dimensional vector (a list of floating-point numbers) where the geometry of the vectors captures the semantic meaning of the text.

If the embedding model is trained well, vectors for "car" and "automobile" will be located close to each other in this high-dimensional space, while "banana" will be far away.

### The "How": Sentence Transformers
While Large Language Models (LLMs) like GPT-4 can generate embeddings, it is often more cost-effective and faster in production to use specialized, smaller models for this task. `sentence-transformers` is an industry-standard Python library built on top of PyTorch and Hugging Face Transformers. It provides access to models like `all-MiniLM-L6-v2`, which are optimized for generating sentence and paragraph embeddings quickly.

### Measuring Distance: Cosine Similarity
Once we have vectors, we need a way to compare them. **Cosine Similarity** measures the cosine of the angle between two non-zero vectors. 
- A similarity of **1.0** means the vectors point in the exact same direction (highly similar semantics).
- A similarity of **0.0** means they are orthogonal (unrelated).
- A similarity of **-1.0** means they are exactly opposite.

Cosine similarity is preferred over Euclidean distance for embeddings because it cares about the *angle* (semantic content) rather than the *magnitude* (length of the text).

### AI Security & Production Implications
- **PII Leakage:** Embeddings are mathematical representations of text. If you embed text containing Personally Identifiable Information (PII) and store it in a vector database, that PII can potentially be reconstructed or searched by unauthorized users. Always scrub PII *before* generating embeddings.
- **Fallback Mechanisms:** In production, calls to external embedding APIs or loading local models can fail due to OOM (Out of Memory) errors or network issues. Implement robust `try/except` blocks and fallback models (e.g., falling back to a simpler model or API if the primary one fails).


## 2. Code Implementation

We will explore text embeddings and cosine similarity in three tiers: Basic, Medium, and Advanced.

> **Note on Dependencies**: Make sure you have `sentence-transformers` and `numpy` installed.
> `uv pip install sentence-transformers numpy`


### Basic: Core Concept
Isolating the core concept of generating embeddings and computing cosine similarity with minimal boilerplate.

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a simple, fast model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Define two sentences
sentence1 = "The cat sits outside."
sentence2 = "A feline is resting outdoors."

# Generate embeddings (convert to numpy arrays for standard math)
emb1 = model.encode(sentence1, convert_to_numpy=True)
emb2 = model.encode(sentence2, convert_to_numpy=True)

# Compute cosine similarity manually
dot_product = np.dot(emb1, emb2)
norm_1 = np.linalg.norm(emb1)
norm_2 = np.linalg.norm(emb2)
similarity = dot_product / (norm_1 * norm_2)

print(f"Basic Similarity: {similarity:.4f}")


/app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1743.35it/s]

Basic Similarity: 0.6093


### Medium: Clean OOP & State Management
Encapsulating the logic in a class to manage the model's state (so we don't reload the heavy weights on every call).

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List

class DocumentEmbedder:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        # Load model once during initialization
        self.model = SentenceTransformer(model_name)
        
    def embed_documents(self, documents: List[str]) -> np.ndarray:
        return self.model.encode(documents, convert_to_numpy=True)
        
    def calculate_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)
        if norm1 == 0 or norm2 == 0:
            return 0.0
        return float(np.dot(vec1, vec2) / (norm1 * norm2))

# Usage
embedder = DocumentEmbedder()
docs = ["I love programming.", "Coding is my passion."]
embeddings = embedder.embed_documents(docs)
sim = embedder.calculate_similarity(embeddings[0], embeddings[1])
print(f"Medium OOP Similarity: {sim:.4f}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1633.70it/s]

Medium OOP Similarity: 0.7484


### Advanced: Production-Grade Implementation
Production-grade implementation with strict type hinting, docstrings, error handling, AI security considerations (fallback mechanisms), and exact import syntax.

In [3]:
import logging
import numpy as np
from typing import List, Optional
from sentence_transformers import SentenceTransformer
from numpy.typing import NDArray

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ProductionEmbeddingService:
    """
    A production-ready service for generating and comparing text embeddings.
    Includes error handling, logging, and a fallback mechanism.
    """
    
    def __init__(self, primary_model_name: str = "all-MiniLM-L6-v2", fallback_model_name: str = "paraphrase-MiniLM-L3-v2") -> None:
        """
        Initializes the EmbeddingService with a primary and fallback model.
        """
        self.primary_model_name = primary_model_name
        self.fallback_model_name = fallback_model_name
        self.model: Optional[SentenceTransformer] = None
        self._initialize_model()
        
    def _initialize_model(self) -> None:
        """Attempts to load the primary model, falling back to an alternative if it fails."""
        try:
            logger.info(f"Loading primary model: {self.primary_model_name}")
            self.model = SentenceTransformer(self.primary_model_name)
        except Exception as e:
            logger.error(f"Failed to load primary model due to {e}. Attempting fallback...")
            try:
                self.model = SentenceTransformer(self.fallback_model_name)
                logger.info(f"Fallback model loaded: {self.fallback_model_name}")
            except Exception as fallback_e:
                logger.critical(f"Fallback model also failed: {fallback_e}")
                raise RuntimeError("Could not initialize embedding models.") from fallback_e

    def generate_embeddings(self, texts: List[str]) -> NDArray[np.float64]:
        """
        Generates dense vector representations with input validation.
        
        Args:
            texts (List[str]): Strings to embed. PII must be scrubbed prior to calling this!
            
        Returns:
            NDArray[np.float64]: A 2D numpy array of embeddings.
        """
        if not texts:
            logger.warning("Empty list provided to generate_embeddings. Returning empty array.")
            return np.array([])
            
        if not self.model:
             raise RuntimeError("Model is not initialized.")
             
        try:
            # Explicitly request numpy conversion
            embeddings = self.model.encode(texts, convert_to_numpy=True)
            return embeddings
        except Exception as e:
            logger.error(f"Error during embedding generation: {e}")
            raise
            
    def compute_cosine_similarity(self, vec1: NDArray[np.float64], vec2: NDArray[np.float64]) -> float:
        """
        Computes cosine similarity safely.
        """
        if vec1.shape != vec2.shape:
             logger.error("Vector dimension mismatch.")
             raise ValueError("Vectors must have the same dimensions.")
             
        dot_product = np.dot(vec1, vec2)
        norm_a = np.linalg.norm(vec1)
        norm_b = np.linalg.norm(vec2)
        
        if norm_a == 0.0 or norm_b == 0.0:
            logger.warning("One of the vectors has zero magnitude.")
            return 0.0
            
        return float(dot_product / (norm_a * norm_b))

# --- Example Usage ---
if __name__ == "__main__":
    try:
        service = ProductionEmbeddingService()
        
        # NOTE: In a real system, ensure 'texts' are scrubbed of PII before this step!
        sample_texts = [
            "Data security and privacy are paramount.",
            "Protecting sensitive information is crucial."
        ]
        
        embs = service.generate_embeddings(sample_texts)
        if len(embs) == 2:
             sim = service.compute_cosine_similarity(embs[0], embs[1])
             print(f"\n[Advanced] Similarity Score: {sim:.4f}\n")
    except Exception as e:
        logger.error(f"Execution failed: {e}")


INFO:__main__:Loading primary model: all-MiniLM-L6-v2


INFO:sentence_transformers.base.model:No device provided, using cpu


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


...

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/2_Normalize/config.json "HTTP/1.1 404 Not Found"


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 66.18it/s]


[Advanced] Similarity Score: 0.7355



## 3. Common Pitfalls

When moving embedding workloads to production, several issues frequently arise:

1. **Model Mismatch**: If you index data in a vector database using `all-MiniLM-L6-v2` but later query it using embeddings generated by `text-embedding-3-small` (OpenAI), the vector spaces are completely incompatible. Searches will return garbage. **Rule**: Always use the exact same model for generating query embeddings that you used for the document embeddings.
2. **Dimension Mismatch**: Different models output vectors of different lengths (e.g., 384 for MiniLM, 1536 for OpenAI). If your vector database (like Qdrant) is configured for 384 dimensions, inserting a 1536-dimensional vector will throw an error.
3. **Re-initializing the Model in a Loop**: Calling `SentenceTransformer('model_name')` inside an API endpoint or a loop will load the heavy model weights from disk into memory every single time, destroying latency. Always initialize the model once globally or within a class constructor (as demonstrated above).
4. **Context Length Truncation**: Small models often have a maximum sequence length (e.g., 256 or 512 tokens). If you pass a 5,000-word essay into `model.encode()`, it will silently truncate the text and only embed the first paragraph. You must chunk long documents before embedding them.


## 4. Practical Lab / Homework

**Your Task:** Build a Semantic Search function.

Using the `EmbeddingService` class defined above, implement a function that takes a query string and a list of document strings. The function should return the top `k` most similar documents based on cosine similarity.

### Requirements:
1. Implement the `semantic_search` function.
2. Ensure you include strict type hinting and a docstring.
3. Do not use stubs or `pass`; provide a fully working implementation.
4. Test it with the provided query and corpus.


> **Challenge:** Once completed, record a brief 2-3 minute async video walkthrough explaining your design decisions, how you structured the class, and how you would integrate PII scrubbing if deployed to production.


In [4]:
from typing import List, Tuple

def semantic_search(query: str, corpus: List[str], top_k: int = 2) -> List[Tuple[str, float]]:
    """
    Performs a semantic search to find the most relevant documents in a corpus for a given query.
    
    Args:
        query (str): The search query.
        corpus (List[str]): A list of document strings to search through.
        top_k (int): The number of top results to return.
        
    Returns:
        List[Tuple[str, float]]: A list of tuples, where each tuple contains the document string
                                 and its similarity score, sorted in descending order of similarity.
    """
    service = ProductionEmbeddingService()
    
    # 1. Embed the query (returns a 2D array, we need the 1st row)
    query_embedding = service.generate_embeddings([query])[0]
    
    # 2. Embed the corpus
    corpus_embeddings = service.generate_embeddings(corpus)
    
    # 3. Compute similarities
    results = []
    for i, doc_embedding in enumerate(corpus_embeddings):
        sim = service.compute_cosine_similarity(query_embedding, doc_embedding)
        results.append((corpus[i], sim))
        
    # 4. Sort results by similarity score in descending order
    results.sort(key=lambda x: x[1], reverse=True)
    
    # 5. Return top_k
    return results[:top_k]

# --- Test your implementation ---
if __name__ == "__main__":
    search_corpus = [
        "Python is a high-level programming language.",
        "The recipe for a perfect chocolate cake requires high-quality cocoa.",
        "Machine learning models require large amounts of data to train.",
        "Baking bread is an art that requires patience and precise measurements.",
        "Vector databases are designed to store and query high-dimensional embeddings."
    ]
    
    search_query = "What do I need to bake a delicious dessert?"
    
    print(f"Query: '{search_query}'\n")
    top_results = semantic_search(search_query, search_corpus, top_k=2)
    
    print("Top Results:")
    for doc, score in top_results:
        print(f"Score: {score:.4f} | Document: '{doc}'")


INFO:__main__:Loading primary model: all-MiniLM-L6-v2


INFO:sentence_transformers.base.model:No device provided, using cpu


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 80.61it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 32.75it/s]

Top Results:
Score: 0.4893 | Document: 'The recipe for a perfect chocolate cake requires high-quality cocoa.'
Score: 0.4253 | Document: 'Baking bread is an art that requires patience and precise measurements.'


## 5. Reference Links

- [sentence-transformers Documentation](https://sbert.net/)
- [Understanding Cosine Similarity (DeepAI)](https://deepai.org/machine-learning-glossary-and-terms/cosine-similarity)
- [Qdrant: What is a Vector Database?](https://qdrant.tech/articles/what-is-a-vector-database/)